In [1]:
pip install requests beautifulsoup4 tqdm pdfplumber

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
import requests
from bs4 import BeautifulSoup
import pdfplumber
import re

In [2]:
# ================================
# IMPORTS
# ================================
import os
import json
import requests
from bs4 import BeautifulSoup
import pdfplumber
import re


# ================================
# FOLDER SETUP
# ================================
os.makedirs("data/raw_pdfs", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)


# ================================
# CONFIG (ADD LINKS HERE)
# ================================
URLS = [
    "https://www.niehs.nih.gov/health/topics/agents/pesticides",
    "https://eos.com/blog/types-of-fertilizers/",
    "https://www.fertilizer.org/about-fertilizers/types-of-fertilizers/",
    "https://www.fertilizer.org/about-fertilizers/where-are-fertilizers-used/",
    "https://www.fertilizer.org/",
    "https://mausam.imd.gov.in",
    "https://www.nrcs.usda.gov/conservation-basics/natural-resource-concerns/soil",
    "https://agritech.tnau.ac.in/agriculture/agri_nutrientmgt_fertilizers.html",
    "https://gujaratindia.gov.in/Home/Agriculture",
    "https://kshema.co/blogs/essential-tips-for-increasing-crop-yield/",
    "https://aksharfarmtech.com/blog/proven-ways-to-boost-crop-yield-naturally/",
    "https://www.iffco.in/en/organic-and-bio-fertilisers",
    "https://cfqcti.da.gov.in/indfert.html",
    "https://www.kanbiosys.com/bio-fertilizer-names-list/",
    "https://vajiramandravi.com/current-affairs/major-crops-of-india/",
    "https://www.agrifarming.in/district-wise-crop-production-in-gujarat-major-crops-in-gujarat",
    "https://agriasia.in/about-agri-asia-about-gujarat/",
    "https://slbcgujarat.in/state-profile/agriculture/",

    
    
]

PDF_URLS = [
    "https://www.fertilizer.org/wp-content/uploads/2023/01/2016_ifa_reetz.pdf"
]


# ================================
# CLEANING FUNCTION
# ================================
def clean_text(text):
    if not text:
        return ""

    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9.,()%\- ]+', '', text)
    return text.strip()


# ================================
# DOMAIN DETECTION
# ================================
def detect_domain(text):
    text = text.lower()

    if any(word in text for word in ["fertilizer", "urea", "npk"]):
        return "fertilizer"
    elif any(word in text for word in ["soil", "fertility", "nutrient"]):
        return "soil"
    elif any(word in text for word in ["pest", "disease", "insect"]):
        return "pest"
    elif any(word in text for word in ["irrigation", "rain", "weather", "climate"]):
        return "weather"
    elif any(word in text for word in ["crop", "wheat", "rice", "yield"]):
        return "crop"
    else:
        return "general"


# ================================
# CROP DETECTION
# ================================
def detect_crop(text):
    text = text.lower()

    crops = ["wheat", "rice", "maize", "cotton", "sugarcane"]

    for crop in crops:
        if crop in text:
            return crop

    return "general"


# ================================
# KEYWORD EXTRACTION
# ================================
def extract_keywords(text):
    words = text.lower().split()
    important = ["fertilizer", "soil", "pest", "irrigation", "yield"]

    return list(set([w for w in words if w in important]))


# ================================
# SCRAPE PAGE
# ================================
def scrape_page(url):
    try:
        response = requests.get(url, timeout=10)
        if response.status_code != 200:
            return "", None

        soup = BeautifulSoup(response.content, "html.parser")
        paragraphs = soup.find_all("p")

        text = " ".join([p.get_text() for p in paragraphs])
        return clean_text(text), soup

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return "", None


# ================================
# MULTI-PAGE AUTO DETECTION
# ================================
def find_links(base_url, soup, limit=2):
    links = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if href.startswith("/") or base_url in href:
            full_link = href if href.startswith("http") else base_url + href

            if full_link not in links:
                links.append(full_link)

        if len(links) >= limit:
            break

    return links


# ================================
# PDF HANDLING
# ================================
def download_pdf(url, path):
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            with open(path, "wb") as f:
                f.write(response.content)
    except:
        pass


def extract_pdf_text(path):
    text = ""

    try:
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text += page.extract_text() or ""
    except:
        pass

    return clean_text(text)


# ================================
# DATA COLLECTION
# ================================
data = []

print("\n🔹 Collecting data...\n")

for url in URLS:
    print(f"Processing: {url}")

    text, soup = scrape_page(url)

    if text:
        data.append({
            "text": text,
            "source": url,
            "type": "web",
            "domain": detect_domain(text),
            "crop": detect_crop(text),
            "keywords": extract_keywords(text),
            "summary": text[:200]
        })

    # MULTI-PAGE
    if soup:
        links = find_links(url, soup)

        for link in links:
            extra_text, _ = scrape_page(link)

            if extra_text:
                data.append({
                    "text": extra_text,
                    "source": link,
                    "type": "web",
                    "domain": detect_domain(extra_text),
                    "crop": detect_crop(extra_text),
                    "keywords": extract_keywords(extra_text),
                    "summary": extra_text[:200]
                })


# ================================
# PDF PROCESSING
# ================================
for i, pdf_url in enumerate(PDF_URLS):
    path = f"data/raw_pdfs/file_{i}.pdf"

    download_pdf(pdf_url, path)

    text = extract_pdf_text(path)

    if text:
        data.append({
            "text": text,
            "source": pdf_url,
            "type": "pdf",
            "domain": detect_domain(text),
            "crop": detect_crop(text),
            "keywords": extract_keywords(text),
            "summary": text[:200]
        })


# ================================
# SAVE RAW DATA
# ================================
with open("data/processed/data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

print(f"\n✅ Saved {len(data)} documents")


# ================================
# CHUNKING (SMART)
# ================================
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = words[i:i + chunk_size]
        chunks.append(" ".join(chunk))

    return chunks


# ================================
# CREATE CHUNKS WITH METADATA
# ================================
chunks = []
chunk_id = 0

for item in data:
    split_chunks = chunk_text(item["text"])

    for ch in split_chunks:
        chunks.append({
            "chunk_id": chunk_id,
            "text": ch,
            "source": item["source"],
            "type": item["type"],
            "domain": item["domain"],
            "crop": item["crop"],
            "keywords": item["keywords"]
        })
        chunk_id += 1


# ================================
# SAVE CHUNKS
# ================================
with open("data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=4)

print(f"\n✅ Created {len(chunks)} chunks")


# ================================
# PREVIEW
# ================================
print("\n🔹 Sample Chunk:\n")
print(chunks[0])


🔹 Collecting data...

Processing: https://www.niehs.nih.gov/health/topics/agents/pesticides
Processing: https://eos.com/blog/types-of-fertilizers/
Processing: https://www.fertilizer.org/about-fertilizers/types-of-fertilizers/
Processing: https://www.fertilizer.org/about-fertilizers/where-are-fertilizers-used/
Processing: https://www.fertilizer.org/
Processing: https://mausam.imd.gov.in
Processing: https://www.nrcs.usda.gov/conservation-basics/natural-resource-concerns/soil
Processing: https://agritech.tnau.ac.in/agriculture/agri_nutrientmgt_fertilizers.html
Processing: https://gujaratindia.gov.in/Home/Agriculture
Processing: https://kshema.co/blogs/essential-tips-for-increasing-crop-yield/
Processing: https://aksharfarmtech.com/blog/proven-ways-to-boost-crop-yield-naturally/
Processing: https://www.iffco.in/en/organic-and-bio-fertilisers
Processing: https://cfqcti.da.gov.in/indfert.html
Processing: https://www.kanbiosys.com/bio-fertilizer-names-list/
Processing: https://vajiramandravi